# KneeXAI — GPU research runner for Kaggle

This notebook runs the repository's **existing** `scripts/run_research.py` on Kaggle GPU. It does not reimplement the metrics, so the resulting CSV follows the same validated contract used by Research Mode.

## Before running

1. In Kaggle Notebook settings, turn **Accelerator → GPU** on.
2. Attach a Kaggle Dataset containing this project source (without `.venv`, `.git`, and model weights is fine).
3. Attach a second private Kaggle Dataset containing the model files. For the selected model it must contain both the `.pth` checkpoint and matching `.onnx` file listed in the configuration cell.
4. For a thesis run, replace the demo manifest with a labelled JSON manifest for real images. Every `true_kl` and `split` must already be known; this notebook never guesses labels from folder names.
5. Enable Internet only if Kaggle needs to install `quantus` and ONNX packages.

The default values below use the six-image repository demo only as a technical check — not as scientific evidence.

In [ ]:
# Kaggle already provides CUDA-enabled PyTorch when GPU is enabled.
# Do NOT install torch from requirements.txt here: that file is intentionally CPU-oriented.
import sys
!{sys.executable} -m pip install -q quantus==0.6.0 onnx==1.17.0 onnxruntime==1.20.1 opencv-python-headless==4.10.0.84

import torch
assert torch.cuda.is_available(), (
    'CUDA is unavailable. In Kaggle open Notebook options and select Accelerator → GPU, ' 
    'then restart the session and run this cell again.'
)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from datetime import datetime
from pathlib import Path

# CHANGE THESE THREE PATHS to the Kaggle Dataset slugs that you attached.
PROJECT_SOURCE = Path('/kaggle/input/kneexai-project')
MODELS_SOURCE = Path('/kaggle/input/kneexai-models')

# Select exactly one: 'resnet50' or 'densenet121'.
MODEL_NAME = 'resnet50'

# For a real study, point this at your labelled manifest. It may be inside PROJECT_SOURCE
# or another attached dataset. Relative image paths are resolved from the copied project root.
MANIFEST_SOURCE = PROJECT_SOURCE / 'inputs' / 'manifest.json'

# First run: False. If Kaggle disconnects, set True and keep every setting below identical.
RESUME = False
RUN_ID = 'resnet50-demo-' + datetime.now().strftime('%Y%m%d-%H%M%S')

# Full thesis defaults: 50 FaithfulnessCorrelation perturbations / 8 MaxSensitivity samples.
# Use the smaller values only to verify that the pipeline starts.
LIMIT = None                 # e.g. 1 for a technical smoke test
FAITHFULNESS_RUNS = 50       # e.g. 2 for a technical smoke test
MAX_SENSITIVITY_SAMPLES = 8  # e.g. 1 for a technical smoke test
SEED = 42
METHODS = ['Grad-CAM', 'Grad-CAM++', 'Eigen-CAM', 'SmoothGrad', 'Integrated Gradients']

MODEL_FILES = {
    'resnet50': ['resnet50_phase2_best.pth', 'resnet50_kl.onnx'],
    'densenet121': ['densenet121_finetune_block3_4_best.pth', 'densenet121_kl.onnx'],
}
assert MODEL_NAME in MODEL_FILES
assert PROJECT_SOURCE.is_dir(), f'Project dataset not found: {PROJECT_SOURCE}'
assert MODELS_SOURCE.is_dir(), f'Model dataset not found: {MODELS_SOURCE}'
assert MANIFEST_SOURCE.is_file(), f'Manifest not found: {MANIFEST_SOURCE}'
print('Selected model:', MODEL_NAME)
print('Manifest:', MANIFEST_SOURCE)

### Manifest note

The runner validates every image before calculation. If your real manifest is in a separate Kaggle dataset, either use **absolute Kaggle paths** in `image_path`, or copy its images under the project directory and use paths relative to it. Do not create labels from directory names.

In [ ]:
import shutil

PROJECT_DIR = Path('/kaggle/working/knee-osteoarthritis-xai')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(PROJECT_SOURCE, PROJECT_DIR, ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__'))

models_dir = PROJECT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
for filename in MODEL_FILES[MODEL_NAME]:
    source = MODELS_SOURCE / filename
    if not source.is_file():
        raise FileNotFoundError(f'Missing required {MODEL_NAME} model file: {source}')
    shutil.copy2(source, models_dir / filename)

# A source manifest inside the project keeps its relative paths valid after copying.
# External absolute paths remain valid because Kaggle input datasets are mounted read-only.
if MANIFEST_SOURCE.is_relative_to(PROJECT_SOURCE):
    manifest_path = PROJECT_DIR / MANIFEST_SOURCE.relative_to(PROJECT_SOURCE)
else:
    manifest_path = MANIFEST_SOURCE

OUTPUT_PATH = Path('/kaggle/working') / f'{RUN_ID}.csv'
print('Project staged at:', PROJECT_DIR)
print('Output will be:', OUTPUT_PATH)

In [ ]:
# Validate the labelled manifest before spending GPU time.
import json
import os
import subprocess

command = [
    sys.executable, 'scripts/run_research.py',
    '--manifest', str(manifest_path),
    '--model', MODEL_NAME,
    '--methods', *METHODS,
    '--output', str(OUTPUT_PATH),
    '--run-id', RUN_ID,
    '--seed', str(SEED),
    '--faithfulness-runs', str(FAITHFULNESS_RUNS),
    '--max-sensitivity-samples', str(MAX_SENSITIVITY_SAMPLES),
    '--device', 'cuda',
]
if LIMIT is not None:
    command += ['--limit', str(LIMIT)]
if RESUME:
    command.append('--resume')

print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
# Inspect and download the validated result.
import pandas as pd

results = pd.read_csv(OUTPUT_PATH)
display(results.groupby(['metric_category', 'metric_name']).size().rename('rows').reset_index())
display(results.groupby(['true_kl', 'xai_method', 'metric_name'])['metric_value'].agg(['count', 'median', 'mean']).reset_index())
print(f'Saved {len(results)} rows: {OUTPUT_PATH}')
print('In the right Kaggle panel, use Output → Save Version or download this CSV and place it in results/research/ locally.')